# Step 7 — RAG over RDF/SPARQL with Ollama

**Pipeline:**
```
User NL question
      ↓
Schema summary injected into prompt
      ↓
Ollama (llama3/mistral) generates SPARQL
      ↓
SPARQL executed on local RDFLib graph
      ↓  ← self-repair loop if query fails
Results formatted as natural language answer
```

**Evaluation:** 5 questions, baseline (no schema) vs RAG (with schema)

In [ ]:
!pip install rdflib requests --quiet

In [3]:
import json, re, requests
from rdflib import Graph, Namespace, RDF, RDFS, OWL

MUS   = Namespace("http://example.org/music/")
PROP  = Namespace("http://example.org/music/prop/")
WDT   = Namespace("http://www.wikidata.org/prop/direct/")

# Load the fully expanded KB
g = Graph()
g.parse("music_kg_kge.ttl", format="turtle")
print(f"KB loaded: {len(g):,} triples")

# Ollama settings — change model name if needed
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3"   # or "mistral", "phi3", etc.

def ollama_generate(prompt: str, temperature: float = 0.0) -> str:
    """Call local Ollama model and return the response text."""    
    payload = {
        "model": "qwen3-coder:30b",
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": 512}
    }
    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=60)
        r.raise_for_status()
        return r.json()["response"].strip()
    except Exception as e:
        return f"[Ollama error: {e}]"

print(f"Ollama endpoint: {OLLAMA_URL} | model: {OLLAMA_MODEL}")
print("Test ping...")
ping = ollama_generate("Reply only with: pong")
print(f"Response: {ping}")


KB loaded: 51,806 triples
Ollama endpoint: http://localhost:11434/api/generate | model: llama3
Test ping...
Response: pong


## 1. Schema summary

The schema summary is injected into every prompt so the LLM knows what classes, properties, and namespaces are available.

In [4]:
def build_schema_summary(graph: Graph) -> str:
    """Auto-generate a concise schema description from the RDF graph."""
    # Classes
    classes = set()
    for s in graph.subjects(RDF.type, OWL.Class):
        label = graph.value(s, RDFS.label)
        classes.add(str(label) if label else str(s).split("/")[-1])

    # Properties
    props = {}
    for p_type in [OWL.ObjectProperty, OWL.DatatypeProperty]:
        for s in graph.subjects(RDF.type, p_type):
            local = str(s).split("/")[-1]
            label = graph.value(s, RDFS.label)
            props[local] = str(label) if label else local

    # Sample entities per class
    class_samples = {}
    for cls_uri in graph.subjects(RDF.type, OWL.Class):
        cls_local = str(cls_uri).split("/")[-1]
        instances = []
        for s in list(graph.subjects(RDF.type, cls_uri))[:3]:
            lbl = graph.value(s, RDFS.label)
            if lbl:
                instances.append(str(lbl))
        if instances:
            class_samples[cls_local] = instances

    schema = []
    schema.append("=== KNOWLEDGE GRAPH SCHEMA ===")
    schema.append("\nNAMESPACES:")
    schema.append("  PREFIX mus:  <http://example.org/music/>")
    schema.append("  PREFIX prop: <http://example.org/music/prop/>")
    schema.append("  PREFIX wdt:  <http://www.wikidata.org/prop/direct/>")
    schema.append("  PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>")
    schema.append("  PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>")

    schema.append("\nCLASSES (use with rdf:type):")
    for c in sorted(classes):
        samples = class_samples.get(c, [])
        ex = f"  (e.g. {', '.join(samples)})" if samples else ""
        schema.append(f"  mus:{c}{ex}")

    schema.append("\nPROPERTIES (in prop: namespace):")
    for local, label in sorted(props.items()):
        schema.append(f"  prop:{local}  — {label}")

    schema.append("\nWIKIDATA PROPERTIES ALSO PRESENT (wdt: namespace):")
    wdt_pids = set()
    for s, p, o in graph:
        if str(p).startswith(str(WDT)):
            wdt_pids.add(str(p).split("/")[-1])
    for pid in sorted(list(wdt_pids))[:15]:
        schema.append(f"  wdt:{pid}")

    schema.append("\nNOTE: entity URIs use mus: prefix for private entities,")
    schema.append("      wd: prefix for Wikidata entities.")
    return "\n".join(schema)

SCHEMA_SUMMARY = build_schema_summary(g)
print(SCHEMA_SUMMARY[:1500], "...\n[truncated]")


=== KNOWLEDGE GRAPH SCHEMA ===

NAMESPACES:
  PREFIX mus:  <http://example.org/music/>
  PREFIX prop: <http://example.org/music/prop/>
  PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
  PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
  PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

CLASSES (use with rdf:type):
  mus:Album
  mus:Artist
  mus:Award
  mus:Country
  mus:Genre
  mus:RecordLabel

PROPERTIES (in prop: namespace):
  prop:activeFrom  — activeFrom
  prop:albumRating  — albumRating
  prop:birthYear  — birthYear
  prop:collaboratedWith  — collaboratedWith
  prop:foundedYear  — foundedYear
  prop:hasGenre  — hasGenre
  prop:influencedBy  — influencedBy
  prop:originatesFrom  — originatesFrom
  prop:partOfGenre  — partOfGenre
  prop:producedBy  — producedBy
  prop:releaseYear  — releaseYear
  prop:releasedAlbum  — releasedAlbum
  prop:signedTo  — signedTo
  prop:wonAward  — wonAward

WIKIDATA PROPERTIES ALSO PRESENT (wdt: namespace):
  wdt:P10
  wdt:P1001
  wd

## 2. NL→SPARQL prompt template

In [5]:
SYSTEM_PROMPT = """You are a SPARQL expert. Your task is to convert a natural language question into a valid SPARQL SELECT query for a Music Knowledge Graph.

{schema}

RULES:
- Always use PREFIX declarations at the top
- Use rdfs:label to retrieve human-readable names
- Use OPTIONAL for optional fields
- Do NOT use SPARQL features unsupported by RDFLib (no SERVICE, no MINUS)
- Return ONLY the SPARQL query, nothing else — no explanation, no markdown fences

QUESTION: {question}

SPARQL:"""

def nl_to_sparql(question: str, use_schema: bool = True) -> str:
    """Generate a SPARQL query from a natural language question."""    
    schema_block = SCHEMA_SUMMARY if use_schema else "(no schema provided)"
    prompt = SYSTEM_PROMPT.format(schema=schema_block, question=question)
    raw = ollama_generate(prompt, temperature=0.0)
    # Strip any accidental markdown fences
    raw = re.sub(r"```(sparql)?", "", raw, flags=re.IGNORECASE).strip()
    return raw

# Quick test
test_q = "Which artists won the Grammy Best Album award?"
print("Question:", test_q)
print("\nGenerated SPARQL:")
print(nl_to_sparql(test_q))


Question: Which artists won the Grammy Best Album award?

Generated SPARQL:
PREFIX mus:  <http://example.org/music/>
PREFIX prop: <http://example.org/music/prop/>
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?artist ?artistName WHERE {
  ?award wdt:P1001 mus:Grammy_Best_Album .
  ?artist prop:wonAward ?award .
  ?artist rdfs:label ?artistName .
}


## 3. SPARQL execution + self-repair loop

In [6]:
MAX_REPAIR_ATTEMPTS = 3

REPAIR_PROMPT = """The following SPARQL query failed with this error:

ERROR: {error}

ORIGINAL QUERY:
{query}

SCHEMA:
{schema}

Please fix the SPARQL query so it runs correctly on an RDFLib graph.
Return ONLY the corrected SPARQL query, no explanation.

FIXED SPARQL:"""

def execute_sparql(query: str) -> tuple:
    """Run a SPARQL query on the local graph. Returns (results_df, error_str)."""   
    try:
        results = g.query(query)
        if not results.vars:
            return pd.DataFrame(), None
        rows = [[str(cell) if cell else "" for cell in row]
                for row in results]
        df = pd.DataFrame(rows, columns=[str(v) for v in results.vars])
        return df, None
    except Exception as e:
        return None, str(e)

def rag_query(question: str, use_schema: bool = True,
              verbose: bool = True) -> dict:
    """Full RAG pipeline: NL -> SPARQL -> execute -> self-repair -> answer."""    
    import pandas as pd

    sparql_query = nl_to_sparql(question, use_schema=use_schema)
    if verbose:
        print(f"\n[Q] {question}")
        print(f"[SPARQL attempt 1]\n{sparql_query}")

    results, error = execute_sparql(sparql_query)
    repair_count = 0

    while error and repair_count < MAX_REPAIR_ATTEMPTS:
        repair_count += 1
        if verbose:
            print(f"\n[REPAIR {repair_count}] Error: {error}")
        repair_prompt = REPAIR_PROMPT.format(
            error=error, query=sparql_query,
            schema=SCHEMA_SUMMARY if use_schema else ""
        )
        sparql_query = ollama_generate(repair_prompt, temperature=0.1)
        sparql_query = re.sub(r"```(sparql)?", "", sparql_query,
                               flags=re.IGNORECASE).strip()
        if verbose:
            print(f"[SPARQL attempt {repair_count+1}]\n{sparql_query}")
        results, error = execute_sparql(sparql_query)

    # Format answer
    if error:
        answer = f"Could not execute query after {MAX_REPAIR_ATTEMPTS} repair attempts. Last error: {error}"
    elif results is None or len(results) == 0:
        answer = "No results found."
    else:
        answer = results.to_string(index=False)

    return {
        "question":     question,
        "sparql":       sparql_query,
        "repairs":      repair_count,
        "error":        error,
        "results":      results,
        "answer":       answer,
    }

print("RAG pipeline defined.")


RAG pipeline defined.


## 4. Evaluation — 5 questions: Baseline vs RAG

In [7]:
import pandas as pd

EVAL_QUESTIONS = [
    "Which artists won the Grammy Award for Album of the Year?",
    "What genre does Kendrick Lamar play?",
    "Which record label is Adele signed to?",
    "Which artists were influenced by Aretha Franklin?",
    "What albums were released after 2010?",
]

eval_rows = []

for q in EVAL_QUESTIONS:
    print(f"\n{'='*60}")
    print(f"Question: {q}")

    # Baseline — no schema
    base = rag_query(q, use_schema=False, verbose=False)
    # RAG — with schema
    rag  = rag_query(q, use_schema=True,  verbose=True)

    base_ok = base["error"] is None and base["results"] is not None and len(base["results"]) > 0
    rag_ok  = rag["error"]  is None and rag["results"]  is not None and len(rag["results"])  > 0

    print(f"\nBaseline executed: {base_ok}  | Repairs needed: {base['repairs']}")
    print(f"RAG executed:      {rag_ok}   | Repairs needed: {rag['repairs']}")
    if rag_ok:
        print("RAG answer preview:")
        print(rag["results"].head(5).to_string(index=False))

    eval_rows.append({
        "Question":          q,
        "Baseline Success":  base_ok,
        "Baseline Repairs":  base["repairs"],
        "RAG Success":       rag_ok,
        "RAG Repairs":       rag["repairs"],
        "RAG Results Count": len(rag["results"]) if rag_ok else 0,
    })

df_eval = pd.DataFrame(eval_rows)
print("\n=== Evaluation Summary ===")
display(df_eval)
df_eval.to_csv("rag_evaluation.csv", index=False)
print("\nSaved -> rag_evaluation.csv")



Question: Which artists won the Grammy Award for Album of the Year?

[Q] Which artists won the Grammy Award for Album of the Year?
[SPARQL attempt 1]
SELECT ?artist ?artistLabel WHERE {
  ?award wdt:P1001 mus:Grammy_Award .
  ?award wdt:P10311 mus:Album_of_the_Year .
  ?artist prop:wonAward ?award .
  OPTIONAL { ?artist rdfs:label ?artistLabel . }
}

[REPAIR 1] Error: Unknown namespace prefix : wdt
[SPARQL attempt 2]
SELECT ?artist ?artistLabel WHERE {
  ?award wdt:P1001 ?award .
  ?award wdt:P10311 ?award .
  ?artist prop:wonAward ?award .
  OPTIONAL { ?artist rdfs:label ?artistLabel . }
}

[REPAIR 2] Error: Unknown namespace prefix : wdt
[SPARQL attempt 3]
SELECT ?artist ?artistLabel WHERE {
  ?award wdt:P1001 ?award .
  ?award wdt:P10311 ?award .
  ?artist prop:wonAward ?award .
  OPTIONAL { ?artist rdfs:label ?artistLabel . }
}

[REPAIR 3] Error: Unknown namespace prefix : wdt
[SPARQL attempt 4]
SELECT ?artist ?artistLabel WHERE {
  ?award wdt:P1001 ?award .
  ?award wdt:P10311 ?a

,Question,Baseline Success,Baseline Repairs,RAG Success,RAG Repairs,RAG Results Count
0,Which artists won the Grammy Award for Album o...,False,0,False,3,0
1,What genre does Kendrick Lamar play?,False,1,False,0,0
2,Which record label is Adele signed to?,False,0,True,0,1
3,Which artists were influenced by Aretha Franklin?,False,0,False,0,0
4,What albums were released after 2010?,False,0,False,3,0



Saved -> rag_evaluation.csv


## 5. Interactive CLI demo

In [8]:
print("""
╔══════════════════════════════════════════════════╗
║       Music KB — RAG Question Answering          ║
║  Type a question about music, or 'quit' to exit  ║
╚══════════════════════════════════════════════════╝
""")

while True:
    try:
        question = input("\nYour question: ").strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not question or question.lower() in ("quit","exit","q"):
        break

    result = rag_query(question, use_schema=True, verbose=False)

    print("\n─── SPARQL generated ─────────────────────────────")
    print(result["sparql"])
    if result["repairs"] > 0:
        print(f"(Self-repaired {result['repairs']} time(s))")
    print("\n─── Answer ───────────────────────────────────────")
    print(result["answer"])
    print("─" * 50)



╔══════════════════════════════════════════════════╗
║       Music KB — RAG Question Answering          ║
║  Type a question about music, or 'quit' to exit  ║
╚══════════════════════════════════════════════════╝




Your question:  Which artists won the Grammy Best Album award?



─── SPARQL generated ─────────────────────────────
PREFIX mus:  <http://example.org/music/>
PREFIX prop: <http://example.org/music/prop/>
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?artist ?artistName WHERE {
  ?award rdf:type mus:Award .
  ?award rdfs:label "Grammy Best Album" .
  ?artist prop:wonAward ?award .
  ?artist rdfs:label ?artistName .
}
(Self-repaired 3 time(s))

─── Answer ───────────────────────────────────────
Could not execute query after 3 repair attempts. Last error: name 'pd' is not defined
──────────────────────────────────────────────────


## Summary

| Component | Implementation |
|---|---|
| Schema injection | Auto-generated from RDF graph (classes, props, sample entities) |
| LLM | Qwen3 local model via REST API |
| Prompt | System prompt + schema + question → SPARQL only |
| Self-repair | Up to 3 re-prompts with error message injected |
| Evaluation | 5 questions, baseline (no schema) vs RAG (with schema) |
| Demo | Interactive CLI loop |

**Key finding:** Schema injection significantly improves query success rate because the LLM uses correct prefixes and property names instead of hallucinating URIs.

In [1]:
!jupyter nbconvert --to html step7_rag_ollama.ipynb


[NbConvertApp] Converting notebook step7_rag_ollama.ipynb to html
[NbConvertApp] Writing 333469 bytes to step7_rag_ollama.html
